# 01 · Línea base clásica (TF-IDF + Regresión logística)
Referencia **no-deep-learning** para medir cuánto aportan los transformers. No es el modelo final (el enunciado exige DL).
Validación: 5-fold estratificado; se reporta accuracy por idioma.

In [1]:
import sys, re, numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, '../src')
from lang_detect import detect_lang
train = pd.read_csv('../data/train.csv'); ev = pd.read_csv('../data/eval.csv')
for d in (train, ev): d['lang'] = d.text.map(detect_lang); d['n_chars'] = d.text.str.len()
LANGS = ['es', 'en', 'pt', 'fr', 'es-en']; LABELS = ['negativo', 'neutral', 'positivo']
print(train.shape, ev.shape)

(20466, 5) (6781, 4)


In [2]:
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
X = sparse.hstack([TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit_transform(train.text),
                   TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=3, sublinear_tf=True, max_features=150000).fit_transform(train.text)]).tocsr()
pred = cross_val_predict(LogisticRegression(C=4, max_iter=3000), X, train.label, cv=StratifiedKFold(5, shuffle=True, random_state=42), n_jobs=-1)
train['ok'] = pred == train.label
print(f'CV accuracy: {train.ok.mean():.4f}')
display(train.groupby('lang').ok.agg(['size', 'mean']).reindex(LANGS).round(4).rename(columns={'size': 'n', 'mean': 'accuracy'}))

CV accuracy: 0.7201


,n,accuracy
lang,,
es,13649,0.7399
en,2345,0.6832
pt,2305,0.6842
fr,2093,0.6789
es-en,74,0.5270


**Lectura:** ~0.72 global, y cae en en/pt/fr (menos datos y n-gramas que no se comparten entre idiomas). Los n-gramas no
capturan negaciones de largo alcance ni contraste → justifica un transformer multilingüe + aumentación por traducción.